# Example: A Position-Aware Markov Protein Generator
Protein families contain related amino-acid sequences that preserve some positions while allowing others to vary. Can a small family alignment provide enough information to generate new candidate sequences?

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * __Represent an alignment as a matrix:__ A one-hot encoding turns each aligned sequence into a row of indicator values, so a family becomes a data matrix whose column means are its position profile. We'll factor that matrix with the singular value decomposition and read what it says about the family's structure.
> * __Build and sample a position-aware Markov model:__ The model uses a separate amino-acid transition matrix at each pair of adjacent alignment positions. We'll estimate those probabilities from the family and draw a reproducible collection of new sequences.
> * __Evaluate and condition the generator:__ Family statistics tell us what the generated sequences keep and what they lose, including the long-range couplings the matrix picture exposes. We'll measure composition, novelty, identity, diversity, scaffold retention, and the response to sequence weighting.

In this example, we borrow the alignment-to-generation workflow from recent studies of [training-free stochastic attention](https://arxiv.org/abs/2603.14717) and [multiplicity-conditioned protein generation](https://arxiv.org/abs/2603.20115). Our generator is deliberately simpler: it is a first-order Markov model that captures adjacent, position-specific amino-acid relationships.

Let's get started!
___


## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, loads the word-generator exercise and the complete protein-generator module, and imports the packages used here.

Let's set up our code environment:


In [1]:
# Load packages, paths, and the protein-generator module from the local setup file -
# `@__DIR__` is the folder holding this notebook, so the path works no matter which
# directory Jupyter was started from. Re-run this cell after editing anything in src/.
include(joinpath(@__DIR__, "Include.jl"));

The protein generator is implemented in [`src/ProteinMarkov.jl`](src/ProteinMarkov.jl), separate from the student work in `src/Compute.jl`. [The local protein-model reference](docs/protein_markov.md) documents the FASTA reader, model-fitting and sampling functions, and evaluation metrics used below.


### Data

We use the Kunitz-domain alignment from the [stochastic-attention protein-generation study](https://github.com/varnerlab/SA-Protein-Modeling-Study). Kunitz domains are short proteins with a cysteine framework that supports three disulfide bonds.

[The `read_fasta(...)` function](docs/protein_markov.md) reads the records, and [the `standard_alignment(...)` function](docs/protein_markov.md) keeps the ones written entirely in the 20 standard amino acids. Every record here is already the same length, so the ones that drop out carry a gap or a non-standard symbol. They stay in the file for provenance.

Let's load the alignment:

In [2]:
# locate the vendored alignment -
data_path = joinpath(@__DIR__, "data", "PF00014-kunitz-alignment.fasta"); # the FASTA shipped with this example

# read every record, then keep the ones we can model -
protein_records = L11bProteinMarkov.read_fasta(data_path); # Vector of (id, sequence) pairs, one per FASTA header
protein_alignment = L11bProteinMarkov.standard_alignment(protein_records); # Vector{String}: only the 20 standard amino acids
number_of_excluded_records = length(protein_records) - length(protein_alignment); # how many carried a gap or odd symbol

# report what survived the filter -
println("Loaded $(length(protein_records)) aligned Kunitz records.");
println("Retained $(length(protein_alignment)) standard-amino-acid sequences and excluded $(number_of_excluded_records).");
println("Each retained sequence has $(length(first(protein_alignment))) alignment positions.");

Loaded 99 aligned Kunitz records.
Retained 83 standard-amino-acid sequences and excluded 16.
Each retained sequence has 53 alignment positions.


Let's look at the first few retained sequences:

In [3]:
# put the first few aligned sequences into a table -
alignment_preview = DataFrame(
    index = 1:5, # a row counter, so the table is easy to talk about
    sequence = first(protein_alignment, 5), # first(collection, n) takes the leading n entries
);

# show it. `display_size = (-1, -1)` means do not truncate rows or columns -
pretty_table(
    alignment_preview;
    backend = :text,
    display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact),
);

 ------- -------------------------------------------------------
  index                                                sequence 
  Int64                                                  String 
 ------- -------------------------------------------------------
      1   FCLSARDSGPCNNFEKRYGYDANTDTCVEYQYGGCEGTLNNFHSLQRCTEICC
      2   ICLEDVDPGPCQYYQVQWFWDKQVEECKEFHYGGCMGTKNRFSSKQQCVKQCK
      3   VCEMPKETGPCLAYFLHWWYDKKDNTCSMFVYGGCQGNNNNFQSKANCLNTCK
      4   KCLQPVEPGPCKNFADRWYFNVDDGTCHPFKYGGCAGNRNHFFTQKECEVHCA
      5   ACNLPIVQGPCRSYIQLWAFDAVKGKCVRFSYGGCKGNGNKFYSEKECKEYCG
 ------- -------------------------------------------------------


___

## Task 1: Represent the Alignment as a Matrix

In this task, we turn the alignment into a numerical matrix and use the singular value decomposition to ask what structure the family actually has. Everything that follows depends on this choice of representation.

A sequence is a string, and we cannot take the SVD of a string. We encode each alignment position with a __one-hot__ block: a vector of $q=20$ entries that is $1$ for the amino acid present at that position and $0$ for the other nineteen. Writing the blocks of a sequence one position after another gives a row of length $L\cdot{q}$, and stacking the $n$ sequences gives the data matrix

$$
\mathbf{X}\in\mathbb{R}^{n\times{Lq}},
\qquad
X_{k,(\ell-1)q+i} =
\begin{cases}
1 & \text{sequence } k \text{ has amino acid } i \text{ at position } \ell,\\
0 & \text{otherwise.}
\end{cases}
$$

Every row has exactly $L$ nonzero entries, one per position. [The `onehot_matrix(...)` function](docs/protein_markov.md) builds $\mathbf{X}$. Let's encode the alignment:

In [ ]:
# encode the alignment as a one-hot data matrix -
X = L11bProteinMarkov.onehot_matrix(protein_alignment); # n sequences by L*q columns, entries 0.0 or 1.0
number_of_sequences, number_of_columns = size(X); # size returns (rows, columns); unpack both at once
alphabet_size = length(L11bProteinMarkov.AMINO_ACIDS); # q = 20 standard amino acids

# confirm the shape is what the encoding promises -
# `sum(X, dims = 2)` adds across each row and keeps the result a matrix, so `unique(...)`
# returning a single value means every sequence contributed exactly one 1.0 per position.
println("Data matrix X: $(number_of_sequences) sequences by $(number_of_columns) columns.");
println("Columns = positions times alphabet size: $(number_of_columns) = $(length(first(protein_alignment))) x $(alphabet_size).");
println("Nonzero entries per row: $(unique(sum(X, dims = 2))) (one amino acid per position).");

__What is a column mean?__ Column $(\ell-1)q+i$ of $\mathbf{X}$ is an indicator for amino acid $i$ at position $\ell$, so averaging that column over the sequences gives the family frequency of that amino acid at that position. The vector of column means is exactly the __position profile__ of the family, the quantity a profile model stores. Centering subtracts it:

$$
\tilde{\mathbf{X}} = \mathbf{X} - \mathbf{1}\,\bar{\mathbf{x}}^{\top},
\qquad
\bar{\mathbf{x}} = \frac{1}{n}\mathbf{X}^{\top}\mathbf{1}.
$$

What remains in $\tilde{\mathbf{X}}$ is how the sequences differ from the average sequence. Let's center the matrix and confirm the column means are the profile:

In [ ]:
# the column means are the position profile, and centering removes them -
# `mean(X, dims = 1)` averages down each column and returns a 1 by (L*q) matrix;
# `vec(...)` flattens that into a plain vector so we can index it with one number.
profile = vec(mean(X, dims = 1)); # the family frequency of each (position, amino acid) pair
X_centered = X .- profile'; # profile' is a row vector, so the .- subtracts it from every row

# spot-check one entry, then check every entry at once -
# onehot_column(...) gives the column holding a (position, amino acid) pair, and
# position_frequency(...) counts that same pair straight from the alignment.
test_position = 18; # a position where several amino acids occur, so the number is not trivially 0 or 1
test_residue = 'Y';
largest_disagreement = maximum( # the worst mismatch anywhere in the profile; should be exactly 0.0
    abs(profile[L11bProteinMarkov.onehot_column(position, residue)] -
        L11bProteinMarkov.position_frequency(protein_alignment, position, [residue]))
    for position ∈ 1:length(first(protein_alignment)), # this double `for` walks every (position, residue) pair
        residue ∈ L11bProteinMarkov.AMINO_ACIDS);

println("Profile entry from the matrix: $(round(profile[L11bProteinMarkov.onehot_column(test_position, test_residue)], digits = 4))");
println("Same frequency counted directly: $(round(L11bProteinMarkov.position_frequency(protein_alignment, test_position, [test_residue]), digits = 4))");
println("Largest disagreement over all positions and amino acids: $(largest_disagreement)");

Now we factor the centered matrix. [The `svd(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.svd) returns $\tilde{\mathbf{X}}=\mathbf{U}\mathbf{S}\mathbf{V}^{\top}$, with singular values $\sigma_{1}\ge\sigma_{2}\ge\cdots\ge0$ on the diagonal of $\mathbf{S}$. As in the dimensionality-reduction material, the right singular vectors are the principal directions of the data and $\sigma_{i}^{2}$ is proportional to the variance along direction $i$, so

$$
f_{i} = \frac{\sigma_{i}^{2}}{\sum_{j}\sigma_{j}^{2}}
$$

is the fraction of the total variance that direction $i$ carries.

__Prediction:__ A grayscale photograph is close to a low-rank matrix, so a few singular values carry most of its energy. Will a protein family alignment behave the same way?

In [ ]:
# factor the centered data matrix -
# svd(...) returns an object holding U, S, and V, where X_centered ≈ U * Diagonal(S) * V'.
factorization = svd(X_centered);
singular_values = factorization.S; # the σᵢ, already sorted from largest to smallest

# turn the singular values into fractions of the total variance -
# variance along direction i is proportional to σᵢ², so dividing by the sum gives a fraction.
explained_variance = singular_values.^2 ./ sum(singular_values.^2); # the .^ and ./ act entry by entry
cumulative_variance = cumsum(explained_variance); # running total: entry i is the variance in the first i directions

# report the shape of the spectrum -
# findfirst(>=(0.5), ...) returns the index of the first entry that reaches one half.
println("rank of the centered matrix: $(rank(X_centered)) (at most n - 1 = $(number_of_sequences - 1))");
println("variance in the first direction: $(round(100*explained_variance[1], digits = 2))%");
println("directions needed for half the variance: $(findfirst(>=(0.5), cumulative_variance))");

Let's plot how the variance accumulates as we add directions:

In [ ]:
# plot the explained variance, per direction and cumulative -
# a `let` block keeps these names local, so `p` does not leak into the rest of the notebook.
let
    number_of_directions = length(singular_values);

    # an empty plot, then one series at a time -
    p = plot(framestyle = :box, xlabel = "Direction i", ylabel = "Fraction of variance",
        legend = :right, background_color = :transparent,
        background_color_inside = :transparent); # transparent, so it reads in a light or dark editor

    # plot!(...) with the trailing ! adds a series to an existing plot -
    plot!(p, 1:number_of_directions, explained_variance, lw = 2, marker = :circle,
        markersize = 2, label = "per direction"); # how much each single direction carries
    plot!(p, 1:number_of_directions, cumulative_variance, lw = 2, label = "cumulative"); # the running total
    hline!(p, [0.5], lw = 1, ls = :dash, label = "half the variance"); # a horizontal reference line

    p # the last expression is what the cell displays
end

__What do we see?__ The alignment is not close to a low-rank matrix. The centered matrix has the largest rank it could have, one less than the number of sequences, no single direction carries much of the variance, and it takes a sizable fraction of the available directions to reach half of it. The cumulative curve climbs almost as a straight line rather than saturating early.

This is the opposite of the photograph, where the first few rank-one frames rebuild most of the image. Truncating this matrix would discard real family variation, so the Eckart-Young bound tells us there is no good small-$k$ approximation to be had.

That is a statement about the spectrum. The singular __vectors__ are a different matter: each one is a direction in the space of (position, amino acid) pairs, and we can read which pairs it combines. Let's look at the largest entries of $\mathbf{v}_{1}$:

In [ ]:
# the (position, amino acid) pairs with the largest loadings in the first direction -
first_direction = factorization.V[:, 1]; # v₁: column 1 of V, one entry per column of X
number_to_show = 6;

# sort by size regardless of sign, because a large negative loading matters as much as a large positive one -
# sortperm returns the indices that would sort the vector, and rev = true puts the largest first.
loading_order = sortperm(abs.(first_direction); rev = true)[1:number_to_show];

# translate each column index back into a (position, amino acid) pair -
# column j holds amino acid (j - 1) % q + 1 at position (j - 1) ÷ q + 1, inverting the encoding.
# Here ÷ is integer division and % is the remainder.
loading_table = DataFrame(
    position = [(column - 1) ÷ alphabet_size + 1 for column ∈ loading_order],
    amino_acid = [L11bProteinMarkov.AMINO_ACIDS[(column - 1) % alphabet_size + 1]
        for column ∈ loading_order],
    loading = round.(first_direction[loading_order]; digits = 3), # the signed entry of v₁
);

pretty_table(
    loading_table;
    backend = :text,
    display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact),
);

__What do we see?__ The largest loadings do not sit on neighboring positions. The first direction combines columns that are spread along the chain, which says that the amino acid at one position moves together with the amino acid at a position some distance away. Two loadings of opposite sign at the same position simply reflect the one-hot encoding, where exactly one indicator per position is on, but loadings on *different* positions are the interesting part.

Let's measure one of those couplings directly, without the SVD, as a correlation between two indicator columns:

In [ ]:
# the pairs to examine, read off the largest loadings of v₁ above -
# each tuple is (first position, first residue, second position, second residue).
coupled_pairs = [(18, 'Y', 28, 'E'), (18, 'Y', 21, 'N'), (21, 'N', 45, 'L')];

# correlate the two indicator columns of each pair -
# `pair...` splats the tuple, handing its four entries over as separate arguments.
coupling_table = DataFrame(
    pair = ["$(r1)$(l1) with $(r2)$(l2)" for (l1, r1, l2, r2) ∈ coupled_pairs], # a readable label, e.g. "Y18 with E28"
    separation = [abs(l2 - l1) for (l1, _, l2, _) ∈ coupled_pairs], # how far apart along the chain; _ ignores a field
    correlation = [round(L11bProteinMarkov.coupling_correlation(protein_alignment, pair...); digits = 3)
        for pair ∈ coupled_pairs],
);

pretty_table(
    coupling_table;
    backend = :text,
    display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact),
);

__What do we see?__ These positions are genuinely correlated, and they are far apart in the chain. The structure the SVD pointed at is real, and it is not a neighbor effect.

This matters for the rest of the example. The model we are about to build is first order: it ties each position only to the one before it. Nothing in it can represent a dependency that reaches across the chain, so we should expect it to reproduce the profile and the neighbor structure while flattening the couplings we just measured. Task 3 checks whether that is what happens.

___

## Task 2: Build and Sample the Position-Aware Markov Model

In this task, we'll estimate the initial amino-acid distribution and the transition probabilities between every pair of adjacent alignment positions, then draw new sequences from the fitted model.

For amino acids $a$ and $b$, let $n_{\ell}(a,b)$ be the number of aligned family sequences containing $a$ at position $\ell$ and $b$ at position $\ell+1$. The position-specific transition probability is given by:

$$
p_{\ell}(b\mid a)
=
\frac{n_{\ell}(a,b)}
{\sum_{c\in\mathcal{A}}n_{\ell}(a,c)},
$$

where $\mathcal{A}$ is the 20-amino-acid alphabet. The next amino acid depends on the current amino acid and the alignment position. Equivalently, the Markov state is the pair consisting of the position and current amino acid; including the position in the state supplies the information that the next transition needs.

[The `fit_position_aware_markov(...)` function](docs/protein_markov.md) stores the probabilities in a three-dimensional array with one 20×20 matrix for each adjacent pair of positions. Let's fit the model and check its dimensions and probability rows:


In [4]:
# fit the unconditioned family model -
protein_model = L11bProteinMarkov.fit_position_aware_markov(protein_alignment); # every sequence counts equally

# add up each transition row, so we can check the rows are probability distributions -
# transition_probabilities is (L-1) by q by q. Summing over dimension 3 leaves a
# (L-1) by q by 1 array, and dropdims removes that length-1 dimension.
transition_row_sums = dropdims(
    sum(protein_model.transition_probabilities; dims = 3);
    dims = 3,
);
observed_transition_rows = transition_row_sums .> 0; # true where the family actually saw that (position, amino acid)

# check the model against the alignment it came from -
expected_length = length(first(protein_alignment)); # L, the aligned sequence length
expected_alphabet = length(L11bProteinMarkov.AMINO_ACIDS); # q = 20

@testset "Position-aware protein model" begin
    @test protein_model.sequence_length == expected_length
    @test length(protein_model.alphabet) == expected_alphabet
    @test size(protein_model.transition_probabilities) == (expected_length - 1, expected_alphabet, expected_alphabet) # one matrix per adjacent pair
    @test sum(protein_model.initial_probabilities) ≈ 1.0 # ≈ allows for floating-point rounding
    @test all(abs.(transition_row_sums[observed_transition_rows] .- 1.0) .< 1e-12) # every observed row sums to one
    @test all(transition_row_sums[.!observed_transition_rows] .== 0.0) # unobserved rows stay all zeros
end;

Test Summary:                | Pass  Total  Time
Position-aware protein model |    5      5  0.5s


__What does the model contain?__ The initial probability vector covers the first alignment position. The transition array holds one matrix for each adjacent pair of positions, so a sequence of length $L$ needs $L-1$ of them. A row stays zero when its amino acid never appears at that position in the family.

Let's inspect the most common amino acids at the first position:

In [5]:
# rank the initial amino-acid probabilities -
initial_order = sortperm(protein_model.initial_probabilities; rev = true); # indices, most likely first
number_to_show = 8;
initial_indices = initial_order[1:number_to_show]; # keep only the leading few

# pair each index with its amino acid and probability -
initial_probability_table = DataFrame(
    amino_acid = protein_model.alphabet[initial_indices], # indexing with a vector picks several entries at once
    probability = round.(protein_model.initial_probabilities[initial_indices]; digits = 4),
);

pretty_table(
    initial_probability_table;
    backend = :text,
    display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact),
);

 ------------ -------------
  amino_acid   probability 
        Char       Float64 
 ------------ -------------
           I        0.1807
           F        0.1446
           V        0.1446
           R        0.0843
           A        0.0723
           T        0.0723
           L        0.0602
           P        0.0602
 ------------ -------------


At every later position, the active row is chosen by the amino acid just generated. Sampling each position on its own would reproduce the column frequencies but lose the link between neighbors.

### Generate Candidate Sequences

With the model fitted, we draw new sequences one amino acid at a time. [The `sample_sequences(...)` function](docs/protein_markov.md) samples the first amino acid from the initial distribution, then uses the transition matrix at each position to draw the next one.

We set a local random seed so the collection is reproducible:

In [6]:
# generation settings -
seed = 5800; # fixes the random draws, so re-running this cell gives the same sequences
number_of_generated_sequences = 1000; # candidate sequences to generate

# sample candidate sequences -
# The sampler draws the first amino acid from the initial distribution, then walks the
# chain, using the transition matrix belonging to each adjacent pair of positions.
generated_sequences = L11bProteinMarkov.sample_sequences(
    protein_model,
    number_of_generated_sequences;
    seed = seed,
);

# check the samples look like family members before we measure anything -
@testset "Generated protein sequences" begin
    @test length(generated_sequences) == number_of_generated_sequences
    @test all(length(sequence) == protein_model.sequence_length for sequence ∈ generated_sequences) # right length
    @test all(all(amino_acid ∈ protein_model.alphabet for amino_acid ∈ sequence) for sequence ∈ generated_sequences) # right alphabet
end;

# Set(...) drops duplicates, so its length counts the distinct candidates -
println("Generated $(length(generated_sequences)) candidate sequences.");
println("Unique candidates: $(length(Set(generated_sequences))).");

Test Summary:               | Pass  Total  Time
Generated protein sequences |    3      3  0.0s
Generated 1000 candidate sequences.
Unique candidates: 1000.


The samples contain only standard amino acids and have the same aligned length as the family data. Let's inspect the first ten:


In [7]:
# put the first few generated sequences into a table -
generated_preview = DataFrame(
    index = 1:10,
    candidate_sequence = first(generated_sequences, 10),
);

pretty_table(
    generated_preview;
    backend = :text,
    display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact),
);

 ------- -------------------------------------------------------
  index                                      candidate_sequence 
  Int64                                                  String 
 ------- -------------------------------------------------------
      1   FCTQPKQQGTCNNFVRRWWYSPKAKKCRRFEYLGCEGNGNQFESQQECDDRCT
      2   VCTQRQEEGVCDRSQRMWFYDVETKQCRRFNYGGCQGNNNNYETKEDCLRRCF
      3   VCLEDAASGSCSVTVLVWYFNPQAKHCHPFLYNGCNGNGNNFEKLEECEQRCV
      4   RCMLSKVQGPCDRSQRRYWYDLKDGKCERFIYYGCGGNLNNFQSKQQCEKMCP
      5   ACSLKPSRGRCKATVNHWWYNSTEGKCKSFVYGGCLGNNNHFASIMACLQKCV
      6   VCTLSNKVGPCKAGLTRWAFDTFSMACEEFTFYGCQGNLNNFGSKQECERLCI
      7   DCLSPKEEGQCSVLFEKYWYDSEKSKCQMFWWRGCDGNTNNYRSKQQCRDACS
      8   KCEQARNEGTCKGSQRRWYHDPVTHSCESFIYTGCGGNLNKFEEELECESSCV
      9   VCSLPKETGFGDKISDSFYFNPITDTCTPFIYNGCGGNQNNFFDYNQCQQSCK
     10   LCVDLPSAGECGAPAVRFFYNAKTRQCNNFHYGGCLGNANNFESQDECKKKCL
 ------- -------------------------------------------------------


__What do we see?__ The sequences keep the cysteine-rich pattern and several recurring Kunitz motifs, yet differ from one another. Each draw uses only the current amino acid and position, so the model carries no direct link between distant positions.

These are computational candidates: a stable fold or any activity would have to be shown experimentally.

___

## Task 3: Evaluate and Condition the Generator

In this task, we'll compare the generated collection with the stored family, then bias the generator toward a designated subset.

Six checks do the work, and they rest on two quantities. For a set $S$ of aligned sequences of length $L$ over the alphabet $\mathcal{A}$, the __pooled composition__ of the set and the __sequence identity__ of two sequences are

$$
p_{S}(a)=\frac{1}{|S|\,L}\sum_{x\in{S}}\sum_{\ell=1}^{L}\mathbb{1}\left[x_{\ell}=a\right],
\qquad
\mathrm{id}(x,y)=\frac{1}{L}\sum_{\ell=1}^{L}\mathbb{1}\left[x_{\ell}=y_{\ell}\right],
$$

where $\mathbb{1}\left[\cdot\right]$ equals one when its condition holds and zero otherwise. The composition sums to one over the alphabet, and the identity lies between zero and one.

Write $F$ for the stored family and $G$ for the generated collection:

| Check | Definition | Reads as |
|---|---|---|
| Composition divergence | $D_{\mathrm{KL}}(p_{F}\parallel{p_{G}})=\sum_{a\in\mathcal{A}}p_{F}(a)\log\dfrac{p_{F}(a)}{p_{G}(a)}$ | zero when the two compositions agree |
| Novelty | $\dfrac{1}{\lvert{G}\rvert}\,\bigl\lvert\{x\in{G}:x\notin{F}\}\bigr\rvert$ | fraction of candidates not already stored |
| Nearest-family identity | $\dfrac{1}{\lvert{G}\rvert}\displaystyle\sum_{x\in{G}}\max_{y\in{F}}\,\mathrm{id}(x,y)$ | how close candidates sit to the family |
| Pairwise diversity | $1-\operatorname{mean}_{\{x,x^{\prime}\}\subset{G}}\,\mathrm{id}(x,x^{\prime})$ | how different candidates are from each other |
| Scaffold retention | $\dfrac{1}{\lvert{G}\rvert}\,\bigl\lvert\{x:x_{\ell}=\mathtt{C}\;\;\forall\ell\in\mathcal{C}_{\tau}\}\bigr\rvert$ | fraction keeping every conserved cysteine |
| Marker frequency | $\dfrac{1}{\lvert{G}\rvert}\,\bigl\lvert\{x:x_{\ell^{\star}}\in\mathcal{R}\}\bigr\rvert$ | fraction carrying a contact residue |

The last two use the family profile from Task 1. Writing $p_{F,\ell}(a)$ for the frequency of amino acid $a$ at position $\ell$, the scaffold columns are $\mathcal{C}_{\tau}=\{\ell:p_{F,\ell}(\mathtt{C})\ge\tau\}$ for a threshold $\tau$, while $\ell^{\star}$ is the trypsin-contact position and $\mathcal{R}=\{\mathtt{K},\mathtt{R}\}$ the residues we condition on.

Two details matter when reading the numbers. The divergence uses $\max(p_{G}(a),\varepsilon)$ in the denominator for a small $\varepsilon$, so an amino acid the generator never emits gives a large finite value instead of an infinite one. The diversity is estimated from a bounded number of sampled pairs rather than every pair. [The evaluation functions](docs/protein_markov.md) calculate each quantity.

Let's compute the unconditioned result:

In [8]:
# identify the family scaffold and marker -
cysteine_threshold = 0.85; # a column counts as scaffold if at least this fraction of the family has C there
cysteine_positions = L11bProteinMarkov.conserved_positions( # the 𝒞τ of the Task 3 definitions
    protein_alignment,
    'C';
    threshold = cysteine_threshold,
);
marker_position = 25; # ℓ⋆, the aligned primary-contact position
marker_residues = ['K', 'R']; # ℛ, the basic residues defining the designated subset

# quantities the metrics below reuse -
stored_sequence_set = Set(protein_alignment); # a Set makes the "is this sequence already known" test fast
reference_composition = L11bProteinMarkov.amino_acid_composition(protein_alignment); # p_F, sums to one
generated_composition = L11bProteinMarkov.amino_acid_composition(generated_sequences); # p_G, sums to one

# composition divergence: D_KL(p_F ‖ p_G), zero when the two compositions agree -
unconditioned_composition_divergence = L11bProteinMarkov.kl_divergence(
    reference_composition,
    generated_composition,
);

# nearest-family identity: average over candidates of the best match in the family -
unconditioned_nearest_identity = L11bProteinMarkov.mean_nearest_identity(
    generated_sequences,
    protein_alignment,
);

# pairwise diversity: one minus the mean identity between candidates -
unconditioned_diversity = L11bProteinMarkov.pairwise_diversity(generated_sequences);

# scaffold retention: fraction of candidates keeping C at every scaffold column -
unconditioned_scaffold_retention = L11bProteinMarkov.scaffold_retention(
    generated_sequences,
    cysteine_positions,
);

# marker frequency: fraction of candidates carrying K or R at the contact position -
unconditioned_marker_frequency = L11bProteinMarkov.position_frequency(
    generated_sequences,
    marker_position,
    marker_residues,
);

# novelty: fraction of candidates that are not already stored sequences -
# count(f, collection) counts the entries for which f returns true; ∉ means "not in".
unconditioned_novel_fraction = count(
    sequence -> sequence ∉ stored_sequence_set,
    generated_sequences,
) / length(generated_sequences);

println("Six-cysteine scaffold positions: $(cysteine_positions)");
println("Stored-family marker frequency: $(round(100 * L11bProteinMarkov.position_frequency(protein_alignment, marker_position, marker_residues), digits = 1))%");

Six-cysteine scaffold positions: [2, 11, 27, 35, 48, 52]
Stored-family marker frequency: 31.3%


The first generator weights every family sequence equally. To condition it, we give the sequences carrying lysine or arginine at the marker position a larger weight and leave the rest at one. This mirrors the one-ratio conditioning idea in the multiplicity study, except that our weights scale the Markov counts directly.

Let's fit the weighted model and generate a second collection, reusing the seed:

In [9]:
# conditioning settings -
multiplicity_ratio = 20.0; # how much more a marker-positive sequence counts than any other
# the ternary `condition ? a : b` picks a when the condition holds and b otherwise,
# so this gives every marker-positive sequence the larger weight and the rest a weight of one.
sequence_weights = [
    sequence[marker_position] ∈ marker_residues ? multiplicity_ratio : 1.0
    for sequence ∈ protein_alignment
];

# fit a second model from the same alignment, now with weighted counts -
conditioned_model = L11bProteinMarkov.fit_position_aware_markov(
    protein_alignment;
    weights = sequence_weights,
);

# sample from it, reusing the seed so the comparison is reproducible -
conditioned_sequences = L11bProteinMarkov.sample_sequences(
    conditioned_model,
    number_of_generated_sequences;
    seed = seed,
);

# the same six metrics as before, now on the conditioned collection -
conditioned_composition = L11bProteinMarkov.amino_acid_composition(conditioned_sequences);
conditioned_composition_divergence = L11bProteinMarkov.kl_divergence(
    reference_composition, # still compared against the family, not against the other generator
    conditioned_composition,
);
conditioned_nearest_identity = L11bProteinMarkov.mean_nearest_identity(
    conditioned_sequences,
    protein_alignment,
);
conditioned_diversity = L11bProteinMarkov.pairwise_diversity(conditioned_sequences);
conditioned_scaffold_retention = L11bProteinMarkov.scaffold_retention(
    conditioned_sequences,
    cysteine_positions,
);
conditioned_marker_frequency = L11bProteinMarkov.position_frequency(
    conditioned_sequences,
    marker_position,
    marker_residues,
);
conditioned_novel_fraction = count(
    sequence -> sequence ∉ stored_sequence_set,
    conditioned_sequences,
) / length(conditioned_sequences);

We now compare the stored family, the unconditioned generator, and the conditioned generator:


In [10]:
# collect the comparison metrics -
# One NamedTuple per source, all with the same field names, so DataFrame stacks them as rows.
# The stored family is the yardstick: it is identical to itself, so its divergence is zero,
# its novelty is zero, and its nearest-family identity is one by definition.
comparison_table = DataFrame([
    (
        source = "Stored family",
        composition_KL = 0.0,
        novel_percent = 0.0,
        nearest_identity = 1.0,
        pairwise_diversity = L11bProteinMarkov.pairwise_diversity(protein_alignment),
        six_C_scaffold_percent = 100 * L11bProteinMarkov.scaffold_retention( # not 100%: the threshold admits columns a few sequences break
            protein_alignment,
            cysteine_positions,
        ),
        marker_KR_percent = 100 * L11bProteinMarkov.position_frequency(
            protein_alignment,
            marker_position,
            marker_residues,
        ),
    ),
    (
        source = "Unconditioned Markov",
        composition_KL = unconditioned_composition_divergence,
        novel_percent = 100 * unconditioned_novel_fraction,
        nearest_identity = unconditioned_nearest_identity,
        pairwise_diversity = unconditioned_diversity,
        six_C_scaffold_percent = 100 * unconditioned_scaffold_retention,
        marker_KR_percent = 100 * unconditioned_marker_frequency,
    ),
    (
        source = "Conditioned Markov",
        composition_KL = conditioned_composition_divergence,
        novel_percent = 100 * conditioned_novel_fraction,
        nearest_identity = conditioned_nearest_identity,
        pairwise_diversity = conditioned_diversity,
        six_C_scaffold_percent = 100 * conditioned_scaffold_retention,
        marker_KR_percent = 100 * conditioned_marker_frequency,
    ),
]);

# round every column except the first, which holds the source names -
for column_name ∈ names(comparison_table)[2:end]
    comparison_table[!, column_name] = round.(comparison_table[!, column_name]; digits = 3); # df[!, name] is that column
end

pretty_table(
    comparison_table;
    backend = :text,
    display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact),
);

 ---------------------- ---------------- --------------- ------------------ -------------------- ------------------------ -------------------
                source   composition_KL   novel_percent   nearest_identity   pairwise_diversity   six_C_scaffold_percent   marker_KR_percent 
                String          Float64         Float64            Float64              Float64                  Float64             Float64 
 ---------------------- ---------------- --------------- ------------------ -------------------- ------------------------ -------------------
         Stored family              0.0             0.0                1.0                0.624                   87.952              31.325
  Unconditioned Markov              0.0           100.0              0.505                 0.62                     76.1                31.5
    Conditioned Markov            0.005           100.0              0.526                0.597                     61.9                88.7
 --------

__What do we see?__ Every generated sequence is new. The unconditioned collection closely matches the pooled composition, stays only partly identical to its nearest stored sequence, and keeps the cysteine scaffold in most candidates. Read that scaffold number against the stored family rather than against perfect retention: the threshold admits columns that a few family sequences already break.

Weighting the marker-positive subset raises the marker frequency well above the family value, and it lowers diversity and scaffold retention while raising composition divergence and nearest-family identity. Moving the generator toward one marker therefore changes other family properties, which is why a conditioned generator needs several checks rather than one.

None of those six checks looks along the chain. Task 1 found positions far apart that move together, and a first-order model ties each position only to the one before it, so we add a seventh check. [__Coupling recovery__](docs/protein_markov.md) is the correlation between two indicator columns,

$$
r\left[(\ell_{1},a_{1}),(\ell_{2},a_{2})\right]
=\operatorname{cor}\left(\mathbb{1}\left[x_{\ell_{1}}=a_{1}\right],\,\mathbb{1}\left[x_{\ell_{2}}=a_{2}\right]\right),
$$

taken over the sequences of a collection. We measure it on the same pairs we checked in the family:

In [ ]:
# coupling recovery: the Task 1 correlations, measured in each collection -
# one row per pair, one column per collection, using the same function as Task 1.
coupling_recovery = DataFrame(
    pair = ["$(r1)$(l1) with $(r2)$(l2)" for (l1, r1, l2, r2) ∈ coupled_pairs],
    separation = [abs(l2 - l1) for (l1, _, l2, _) ∈ coupled_pairs], # distance along the chain
    family = [round(L11bProteinMarkov.coupling_correlation(protein_alignment, pair...); digits = 3)
        for pair ∈ coupled_pairs],
    unconditioned = [round(L11bProteinMarkov.coupling_correlation(generated_sequences, pair...); digits = 3)
        for pair ∈ coupled_pairs],
    conditioned = [round(L11bProteinMarkov.coupling_correlation(conditioned_sequences, pair...); digits = 3)
        for pair ∈ coupled_pairs],
);

pretty_table(
    coupling_recovery;
    backend = :text,
    display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact),
);

__What do we see?__ The couplings are present in the family and gone from both generated collections. A first-order chain reaches exactly one position ahead, so even the closest pair here, three positions apart, is already beyond what it can represent. The model reproduces the profile, the neighbor structure, and most of the cysteine scaffold, and still loses every one of these agreements.

This is the check the first six miss. A generated collection can match the composition, stay novel, hold the scaffold, and still be wrong in a way that only shows up when we look along the chain. It is also why the singular values alone would not have caught it: the spread of variance across directions is a property the generated collection can imitate, while the directions themselves are not.

___

## Summary

In this example, we represented a protein-family alignment as a matrix, read its structure with the singular value decomposition, built a position-aware Markov model from the same alignment, and measured what the generated sequences kept and what they lost.

> __Key Takeaways:__
>
> * __The representation came before the model:__ We encoded each aligned sequence as a row of one-hot indicators, which makes the column means the family profile and puts the alignment in reach of the singular value decomposition. The spectrum showed that no small set of directions carries the family, and the leading direction combined positions far apart in the chain.
>
> * __A position-aware Markov model produced novel family-like candidates:__ We estimated a transition matrix for every adjacent pair of positions and sampled new sequences one amino acid at a time. Every candidate was new, the pooled composition matched the family, and most candidates kept the conserved cysteines.
>
> * __Conditioning moved one property and disturbed others:__ Weighting the marker-positive sequences raised the marker frequency well above the family value while lowering diversity and scaffold retention. The coupling check exposed a limit that weighting cannot fix, since a first-order chain has nowhere to store agreement between distant positions.

We can now change the weighting, the family alignment, or the random seed and compare how the generated collection responds.

___